# 🐱🐶 Cat vs Dog Classifier — Training Notebook

**Project:** `catdog-vision-api`
**Author:** Dilrabo Khidirova
**Purpose:** This notebook is the *research* layer. We explore the data, build and
train the model, and evaluate it. Once we're happy, the clean logic lives in `src/`
and gets served by the FastAPI app.

> A notebook is for *experiments*. Production logic belongs in `src/`. Keeping them
> separate is exactly what a supervisor looks for in an ML Engineer.

---
## Table of contents
1. Setup & imports
2. Configuration
3. Explore the data
4. Build the model (scratch CNN + ResNet18)
5. Train
6. Evaluate
7. Save weights for the API


In [1]:
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

# Pick GPU if available, else CPU. Everything works on both.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## 2. Configuration
One cell with every setting. No magic numbers scattered through the notebook —
change a value here and the whole notebook follows. This mirrors `configs/config.yaml`.

In [2]:
CONFIG = {
    "data_dir": "../data",          # notebook lives in notebooks/, data is one level up
    "image_size": 224,              # ResNet expects 224x224
    "batch_size": 32,
    "epochs": 5,
    "learning_rate": 1e-4,
    "classes": ["cat", "dog"],      # index 0 = cat, 1 = dog (alphabetical = ImageFolder order)
    "norm_mean": [0.485, 0.456, 0.406],   # ImageNet normalization stats
    "norm_std":  [0.229, 0.224, 0.225],
    "weights_out": "../models/catdog_resnet18.pth",
}
CONFIG

{'data_dir': '../data',
 'image_size': 224,
 'batch_size': 32,
 'epochs': 5,
 'learning_rate': 0.0001,
 'classes': ['cat', 'dog'],
 'norm_mean': [0.485, 0.456, 0.406],
 'norm_std': [0.229, 0.224, 0.225],
 'weights_out': '../models/catdog_resnet18.pth'}

## 3. Explore the data

Expected folder layout (PyTorch's `ImageFolder` reads it automatically — each
subfolder name becomes a label):

```
data/
├── train/cat/*.jpg   train/dog/*.jpg
├── val/cat/*.jpg     val/dog/*.jpg
└── test/
```

Download the Kaggle **Dogs vs. Cats** dataset and arrange it like this, or run
`python scripts/split_data.py` (provided in the repo).

In [3]:
# Count images so we know the dataset is wired up correctly.
for split in ["train", "val"]:
    for cls in CONFIG["classes"]:
        folder = Path(CONFIG["data_dir"]) / split / cls
        n = len(list(folder.glob("*"))) if folder.exists() else 0
        print(f"{split}/{cls:>3}: {n} images")

train/cat: 0 images
train/dog: 0 images
val/cat: 0 images
val/dog: 0 images
